# Mixed Precision Training with PyTorch DDP

This example demonstrates how to train a [ResNet-18](https://arxiv.org/abs/1512.03385) model on the [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset using [PyTorch Automatic Mixed Precision (AMP)](https://pytorch.org/docs/stable/amp.html) with [Distributed Data Parallel (DDP)](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html).

Mixed precision training uses lower-precision floating point types (float16 or bfloat16) for most operations while keeping critical accumulations in float32. This typically halves GPU memory usage and doubles throughput on modern hardware with Tensor Cores.

This notebook shows how to run mixed precision training locally, and how to scale it across multiple nodes with Kubeflow TrainJob.

## Install the Kubeflow SDK

You need to install the Kubeflow SDK to interact with Kubeflow Trainer APIs:

In [ ]:
# !pip install -U kubeflow

## Install the PyTorch Dependencies

You also need to install PyTorch and Torchvision to be able to run the example locally:

In [ ]:
!pip install torch==2.9.1
!pip install torchvision==0.22.1

## Understanding Mixed Precision Training

**Why mixed precision?** Standard training uses float32 (32-bit) for all computations. Mixed precision keeps model weights in float32 but runs forward and backward passes in a lower precision:

- **float16 (fp16)**: Half the memory of float32. Requires a `GradScaler` to prevent gradient underflow because fp16 has a narrow dynamic range (max ~65,504).
- **bfloat16 (bf16)**: Same dynamic range as float32 (max ~3.4e38) but with reduced mantissa precision. Does NOT need a `GradScaler`. Requires Ampere+ GPUs (A100, H100) or recent CPUs.

**Key APIs (PyTorch 2.0+)**:
- `torch.amp.autocast(device_type, dtype)` -- wraps the forward pass to run eligible ops in lower precision.
- `torch.amp.GradScaler(device_type)` -- scales the loss to prevent fp16 gradient underflow, then unscales before the optimizer step. Only needed for fp16.

**Performance impact**:
- Up to 2x faster training on GPUs with Tensor Cores (V100, A100, H100).
- ~50% reduction in GPU memory, enabling larger batch sizes or bigger models.
- bf16 on modern hardware provides the best tradeoff: no scaler complexity, same dynamic range as fp32.

## Define the Training Function

The training function uses ResNet-18 on CIFAR-10 with automatic mixed precision. It detects the available device and selects the appropriate precision dtype and whether to use gradient scaling.

In [ ]:
def train_resnet18_mixed_precision():
    import os
    import time

    import torch
    import torch.distributed as dist
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, DistributedSampler
    from torchvision import datasets, models, transforms

    # --- Device and precision configuration ---
    # Use fp16 + GradScaler on CUDA (works on V100+).
    # Use bf16 without GradScaler on CPU (same dynamic range as fp32).
    if torch.cuda.is_available():
        device_type = "cuda"
        backend = "nccl"
        amp_dtype = torch.float16
        use_grad_scaler = True
    else:
        device_type = "cpu"
        backend = "gloo"
        amp_dtype = torch.bfloat16
        use_grad_scaler = False

    print(f"Device: {device_type}, AMP dtype: {amp_dtype}, GradScaler: {use_grad_scaler}")

    # --- Distributed setup ---
    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    print(
        "Distributed Training for WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )

    device = torch.device(f"{device_type}:{local_rank}" if device_type == "cuda" else device_type)

    # --- Model: ResNet-18 adapted for CIFAR-10 (32x32 images) ---
    model = models.resnet18(num_classes=10)
    # Replace the 7x7 conv with a 3x3 conv (standard for CIFAR-10).
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    # Remove maxpool (not needed for 32x32 input).
    model.maxpool = nn.Identity()
    model = model.to(device)
    model = nn.parallel.DistributedDataParallel(model)

    # --- Optimizer ---
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

    # --- GradScaler for fp16 (prevents gradient underflow) ---
    scaler = torch.amp.GradScaler(device_type) if use_grad_scaler else None

    # --- Dataset: CIFAR-10 with standard augmentation ---
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    # Download dataset only on local_rank=0 to avoid conflicts.
    if local_rank == 0:
        dataset = datasets.CIFAR10("./data", train=True, download=True, transform=transform_train)
    dist.barrier()
    dataset = datasets.CIFAR10("./data", train=True, download=False, transform=transform_train)

    train_loader = DataLoader(
        dataset,
        batch_size=128,
        sampler=DistributedSampler(dataset),
        num_workers=2,
        pin_memory=(device_type == "cuda"),
    )

    # --- Training loop with mixed precision ---
    num_epochs = 3
    dist.barrier()
    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loader.sampler.set_epoch(epoch)
        epoch_start = time.time()
        running_loss = 0.0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            # Autocast wraps the forward pass only.
            with torch.amp.autocast(device_type=device_type, dtype=amp_dtype):
                outputs = model(inputs)
                loss = F.cross_entropy(outputs, labels)

            # Backward pass: use GradScaler for fp16, plain backward for bf16.
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            running_loss += loss.item()

            if batch_idx % 50 == 0 and dist.get_rank() == 0:
                print(
                    "Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.4f}".format(
                        epoch,
                        batch_idx * len(inputs),
                        len(train_loader.dataset),
                        100.0 * batch_idx / len(train_loader),
                        loss.item(),
                    )
                )

        epoch_time = time.time() - epoch_start
        avg_loss = running_loss / len(train_loader)
        if dist.get_rank() == 0:
            print(f"Epoch {epoch} completed in {epoch_time:.1f}s, Average Loss: {avg_loss:.4f}")

    # --- Cleanup ---
    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")
    dist.destroy_process_group()

## Run the Training Locally

We can submit the training function to the local Trainer client to run it in an isolated subprocess.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient, LocalProcessBackendConfig

# Initialize local backend
backend_config = LocalProcessBackendConfig(cleanup_venv=True)
client = TrainerClient(backend_config=backend_config)

# List available runtimes
for runtime in client.list_runtimes():
    if runtime.name == "torch-distributed":
        torch_runtime = runtime
        break

# Submit training job
job_name = client.train(
    trainer=CustomTrainer(
        func=train_resnet18_mixed_precision,
        packages_to_install=["torch", "torchvision"],
    ),
    runtime=torch_runtime,
)

# Stream logs
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end='')

## Scale with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your mixed precision training across multiple PyTorch nodes.

Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in a distributed environment.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient

client = TrainerClient()

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob.

In [ ]:
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

## Run the Distributed TrainJob

The TrainJob will train the ResNet-18 model with mixed precision across 2 PyTorch nodes. Mixed precision halves GPU memory usage, so you can use fewer or smaller GPUs for the same workload.

In [ ]:
job_name = client.train(
    trainer=CustomTrainer(
        func=train_resnet18_mixed_precision,
        # Set how many PyTorch nodes you want to use for distributed training.
        num_nodes=2,
        # Set the resources for each PyTorch node.
        resources_per_node={
            "cpu": 2,
            "memory": "8Gi",
            # Uncomment this to distribute the TrainJob using GPU nodes.
            # "nvidia.com/gpu": 1,
        },
        packages_to_install=["torch", "torchvision"],
    ),
    runtime=torch_runtime,
)

## Check the TrainJob Steps

You can check the components of the TrainJob that was created.

Since the TrainJob performs distributed training across 2 nodes, it generates 2 steps: `trainer-node-0` and `trainer-node-1`.

In [ ]:
# Wait for the running status.
client.wait_for_job_status(name=job_name, status={"Running"})

In [ ]:
for c in client.get_job(name=job_name).steps:
    print(f"Step: {c.name}, Status: {c.status}, Devices: {c.device} x {c.device_count}\n")

## Watch the TrainJob Logs

We can use the `get_job_logs()` API to get the TrainJob logs.

Since we run training on 2 nodes, every PyTorch worker processes 50,000/2 = 25,000 images from the dataset.

In [ ]:
for logline in client.get_job_logs(job_name, follow=True):
    print(logline)

## Delete the TrainJob

When the TrainJob is finished, you can delete the resource.

In [ ]:
# client.delete_job(job_name)